# VGG16 Image Similarity Search

This notebook documents the visual-retrieval experiment used in **La2eny**. A pretrained VGG16 model extracts deep image features, and cosine distance ranks visually similar candidates from a local gallery.

The original dataset and generated feature files are not included. Use your own appropriately licensed images.

## 1. Install dependencies

In [ ]:
!pip install -q tensorflow scipy scikit-learn matplotlib pillow

## 2. Imports and feature extractor

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy.spatial.distance import cosine
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import Model

In [ ]:
# Load ImageNet-pretrained VGG16 and expose its fc2 representation.
base_model = VGG16(weights='imagenet', include_top=True)
feature_extractor = Model(
    inputs=base_model.input,
    outputs=base_model.get_layer('fc2').output
)

## 3. Image preprocessing and feature extraction

In [ ]:
def extract_features(image_path: str) -> np.ndarray:
    img = image.load_img(image_path, target_size=(224, 224))
    array = image.img_to_array(img)
    array = np.expand_dims(array, axis=0)
    array = preprocess_input(array)
    features = feature_extractor.predict(array, verbose=0)[0]
    return features.astype('float32')

## 4. Build the image gallery

In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png'}

def build_image_gallery(folder_path: str):
    folder = Path(folder_path)
    image_paths = sorted(
        p for p in folder.rglob('*')
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )
    if not image_paths:
        raise ValueError(f'No supported images found in: {folder}')

    features = np.vstack([extract_features(str(p)) for p in image_paths])
    return image_paths, features

## 5. Search for similar images

The query image is compared with each gallery feature vector. The closest candidates are returned in ascending cosine-distance order.

In [ ]:
def search_similar_images(query_path: str, image_paths, gallery_features, top_k: int = 5):
    if len(image_paths) == 0 or len(gallery_features) == 0:
        raise ValueError('The image gallery is empty.')

    query_features = extract_features(query_path)
    distances = [cosine(query_features, row) for row in gallery_features]
    ranked_indices = np.argsort(distances)[:top_k]

    return [
        {'path': str(image_paths[i]), 'cosine_distance': float(distances[i])}
        for i in ranked_indices
    ]

## 6. Example usage

Prepare a folder containing your own gallery images and a separate query image.

In [ ]:
# gallery_paths, gallery_features = build_image_gallery('data/image_gallery')
# results = search_similar_images(
#     query_path='data/query_image.jpg',
#     image_paths=gallery_paths,
#     gallery_features=gallery_features,
#     top_k=5,
# )
# for item in results:
#     print(item)

## 7. Visualize retrieved candidates

In [ ]:
def display_results(query_path: str, results):
    count = len(results) + 1
    plt.figure(figsize=(4 * count, 4))

    plt.subplot(1, count, 1)
    plt.imshow(Image.open(query_path).convert('RGB'))
    plt.title('Query image')
    plt.axis('off')

    for position, result in enumerate(results, start=2):
        plt.subplot(1, count, position)
        plt.imshow(Image.open(result['path']).convert('RGB'))
        plt.title(f"Distance: {result['cosine_distance']:.3f}")
        plt.axis('off')

    plt.tight_layout()
    plt.show()

# display_results('data/query_image.jpg', results)

## Notes and limitations

- VGG16 features capture general visual patterns; they are not domain-specific embeddings.
- Cosine distance is a ranking signal, not a calibrated probability of a match.
- The first run downloads pretrained ImageNet weights.
- For large galleries, persist feature vectors and use an approximate nearest-neighbor index.
- Validate retrieval quality with relevant metrics and representative data before deployment.